# HiT-MAC Training — Scenario 1
**Hierarchical Twin-Actor Multi-Agent Coordination for Directional Sensor Networks**

| | |
|---|---|
| **Phase 1** | Executor training — single-att model, A3C, ~5M steps |
| **Phase 2** | Coordinator training — multi-att-shap model, ~5M steps |

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Normal flow:** Run cells 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8  
**If session disconnects during Phase 1:** Run 1 → 2 → 3 → 5R  
**If Phase 1 done, start Phase 2:** Run 1 → 2 → 3 → 6P2 → 7 → 8

In [ ]:
# ── 1. Clone repo ──────────────────────────────────────────────────────────────
import os
if not os.path.exists('DSN_NA2Q'):
    !git clone https://github.com/Chiseled141/DSN_NA2Q.git
if os.path.basename(os.getcwd()) != 'DSN_NA2Q':
    %cd DSN_NA2Q

In [ ]:
# ── 2. Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/DSN_NA2Q/hitmac/scenario1'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted → {DRIVE_DIR}')

In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 4. Verify GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 5. Train Phase 1 — Executor (fresh start) ──────────────────────────────────
# Runs A3C executor training for 5M steps (~50k episodes).
# Phase 2 (coordinator) starts automatically when Phase 1 finishes.
# If you only want Phase 1, interrupt after the Phase 2 banner appears.
!python -m hitmac.main --mode train --scenario 1

In [ ]:
# ── 5R. Resume Phase 1 (if session disconnected during Phase 1) ────────────────
import shutil, os
CKPT_DIR = 'hitmac/checkpoints'
DRIVE_DIR = '/content/drive/MyDrive/DSN_NA2Q/hitmac/scenario1'
PHASE1_FILES = ['latest.pt', 'best.pt', 'training_history.npz', 'training.log']

# Restore Phase 1 checkpoints from Drive
os.makedirs(CKPT_DIR, exist_ok=True)
for fname in PHASE1_FILES:
    src = f'{DRIVE_DIR}/{fname}'
    dst = os.path.join(CKPT_DIR, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'Restored: {fname}')

!python -m hitmac.main --mode train --scenario 1 --resume

In [ ]:
# ── 6P2. Train Phase 2 only — Coordinator (Phase 1 already done) ───────────────
# Run this if Phase 1 is complete (executor_final.pt exists on Drive)
# and you want to start Phase 2 in a new session.
import shutil, os
CKPT_DIR = 'hitmac/checkpoints'
DRIVE_DIR = '/content/drive/MyDrive/DSN_NA2Q/hitmac/scenario1'

# Restore executor from Drive (required for Phase 2)
os.makedirs(CKPT_DIR, exist_ok=True)
for fname in ['executor_final.pt', 'training_history.npz', 'training.log']:
    src = f'{DRIVE_DIR}/{fname}'
    dst = os.path.join(CKPT_DIR, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'Restored: {fname}')

if not os.path.exists(os.path.join(CKPT_DIR, 'executor_final.pt')):
    print('ERROR: executor_final.pt not found on Drive — run Phase 1 first (cell 5)')
else:
    !python -m hitmac.main --mode train --scenario 1 --phase2-only

In [ ]:
# ── 7. Save results to Drive ───────────────────────────────────────────────────
import shutil, os
CKPT_DIR = 'hitmac/checkpoints'
DRIVE_DIR = '/content/drive/MyDrive/DSN_NA2Q/hitmac/scenario1'
SAVE_FILES = [
    'latest.pt',
    'best.pt',
    'executor_final.pt',
    'coordinator_best.pt',
    'coordinator_final.pt',
    'training_history.npz',
    'training.log',
]

saved = []
for fname in SAVE_FILES:
    src = os.path.join(CKPT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_DIR}/{fname}')
        saved.append(fname)

print('Saved to Drive:', saved)

In [ ]:
# ── 8. Download results to your machine ───────────────────────────────────────
from google.colab import files

DRIVE_DIR = '/content/drive/MyDrive/DSN_NA2Q/hitmac/scenario1'
SAVE_FILES = [
    'latest.pt',
    'best.pt',
    'executor_final.pt',
    'coordinator_best.pt',
    'coordinator_final.pt',
    'training_history.npz',
    'training.log',
]

for fname in SAVE_FILES:
    path = f'{DRIVE_DIR}/{fname}'
    if os.path.exists(path):
        files.download(path)